# Stage2_Hazel · 20261026 文本单模态诊断

按《Hazel_明日任务_20261026.md》新增两个设置、共6次训练：Text-only CE，Text-only KD_raw，各三个种子20261010/11/12。

固定 lr=.0003、alpha=.75、tau=2、30 epochs、batch128、AdamW、weight_decay=1e-4。不另搜参数，不做标签平滑、教师重新推理、教师校准或学生后校准。

**Student仅使用文本t；Teacher继续使用图文信息。** 图文CE与原始KD(alpha=.75)从已经完成的Stage2结果包读取，不重复训练。只做val_select开发诊断；不运行历史1000条、补充200条或Winoground。

代码全部内嵌，只需本notebook。这里没有预先填入正式训练成绩；运行后才会生成6个文本模型。


## 1．路径与本次训练配额

`SOURCE`：10月13日完整成果包解压目录，含reused_v1与文件校验清单.json，提供冻结特征和原始Jev训练logits。

`STAGE2_RESULTS`：你已经跑完的正式18条件结果ZIP，例如 Stage2_Hazel_results_f54a0fef.zip；也可以是解压后直接包含PACKAGE_SHA256.json的目录。不是notebook_delivery.zip，也不是教师校准交付ZIP。

`WORK`：全新独立文本实验目录。不要设成上一轮Stage2目录或其子目录。两个输入均只读。

`MAX_NEW_GROUPS=2`完成两个三种子配置；设1则先完成文本CE的三个种子，下次继续KD；设0只整理和导出已有结果。配置配额不改变总共6次训练。


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
SOURCE=Path('/content/drive/MyDrive/1013实验成果包与结论/10月13日_任务交付')
STAGE2_RESULTS=Path('/content/drive/MyDrive/Stage2_Hazel_results_f54a0fef.zip')
WORK=Path('/content/drive/MyDrive/Stage2_Hazel_TextOnly_20261026')
MAX_NEW_GROUPS=2
assert SOURCE.is_dir() and STAGE2_RESULTS.exists(), '请修改SOURCE及STAGE2_RESULTS为实际路径'
for p in [SOURCE,STAGE2_RESULTS]:
    assert not WORK.resolve().is_relative_to(p.resolve()) and not p.resolve().is_relative_to(WORK.resolve()), '输入输出目录必须独立'
WORK.mkdir(parents=True,exist_ok=True)


## 2．检查GPU可用性

这格只检查GPU可用并打印环境。第4节才与Stage2保存的学生环境逐项比较，差异会明确列出；沿用上一轮已记录的Python 3.13.15→3.13.16例外，其他训练环境记录须一致。不会伪造版本或自动升级PyTorch。FP32且关闭TF32。

不需要原图、Teacher或CLIP模型下载，也不需要val_fit图像特征。


In [ ]:
import os,sys,subprocess,importlib.util
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
if importlib.util.find_spec('pandas') is None:
    subprocess.run([sys.executable,'-m','pip','install','pandas'],check=True)
import torch,numpy as np,pandas as pd
assert torch.cuda.is_available(), '请选择GPU运行时'
print('torch:',torch.__version__,'numpy:',np.__version__,'GPU:',torch.cuda.get_device_name(0))


## 3．载入内嵌代码

训练循环沿用Stage2的epoch断点、选模和随机顺序规则，仅将学生输入与头切换为文本方案。输入包内脚本不会被执行。代码变化时不能直接混用旧断点。


In [ ]:
import hashlib,importlib.util
SOURCES={'cache_audit.py': '"""Read-only historical cache audit and disposable, unified training benchmark."""\nfrom pathlib import Path\nimport hashlib, json, time, math, random, platform, uuid, csv\nimport numpy as np\n\nREV = \'3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268\'\nMETHODS = [\'CE\', \'KD_raw\', \'KD_calibrated\', \'KD_base\', \'KD_hard\', \'CE_image\', \'CE_text\']\n\ndef check(ok, message):\n    if not ok: raise ValueError(message)\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for chunk in iter(lambda: f.read(1048576), b\'\'): h.update(chunk)\n    return h.hexdigest()\n\ndef read(path): return json.loads(Path(path).read_text(encoding=\'utf-8-sig\'))\ndef rows(path): return [json.loads(s) for s in Path(path).read_text(encoding=\'utf-8-sig\').splitlines() if s.strip()]\ndef save(path, value):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + \'.\' + uuid.uuid4().hex + \'.partial\')\n    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding=\'utf-8\')\n    tmp.replace(path)\n\ndef locate(source):\n    source = Path(source).resolve()\n    candidates = [source, source/\'reused_v1\']\n    candidates += list(source.glob(\'**/reused_v1\'))\n    matches = sorted(set(p for p in candidates if (p/\'students/training_lock.json\').is_file()))\n    check(len(matches) == 1, \'SOURCE 必须指向已解压的单个成果包或 Student_v1 目录；未唯一找到训练锁。\')\n    return matches[0]\n\ndef audit(source):\n    old = locate(source); lock = read(old/\'students/training_lock.json\')\n    files = {}; data = {}; seen = set(); seen_samples = set()\n    def record(p):\n        digest=sha(p); files[p.relative_to(old).as_posix()] = digest; return digest\n    revision = read(old/\'features/clip_revision.json\')\n    check(revision[\'revision\'] == REV, \'CLIP revision 不符\')\n    record(old/\'features/clip_revision.json\'); record(old/\'students/training_lock.json\')\n    # Only read held-out manifests to verify isolation; never read their features/predictions.\n    for split, n in [(\'train\',6000),(\'val_fit\',400),(\'val_select\',400),(\'test\',1000)]:\n        manifest = old/f\'handoff/manifests/{split}.jsonl\'; rr = rows(manifest); record(manifest)\n        ids = [str(r[\'image_id\']) for r in rr]; sid = [r[\'sample_id\'] for r in rr]\n        check(len(rr)==n and len(set(ids))==n and len(set(sid))==n, f\'{split} 数量/重复 ID 异常\')\n        check(not seen.intersection(ids) and not seen_samples.intersection(sid), \'划分间 ID 泄漏\')\n        seen.update(ids); seen_samples.update(sid)\n        check(all(r[\'label\'] in [0,1,2] and r[\'split\']==split for r in rr), \'标签或 split 异常\')\n        if split not in [\'train\',\'val_select\']: continue\n        p = old/f\'features/{split}.npz\'; meta_path = old/f\'features/{split}_meta.json\'\n        meta = read(meta_path); record(meta_path)\n        digest = record(p)\n        check(digest == meta[\'npz_sha256\'] == lock[\'feature_hashes\'][split], f\'{split} 特征哈希不符\')\n        check(meta[\'manifest_sha256\']==sha(manifest) and meta[\'revision\']==REV, \'特征来源不符\')\n        check(meta[\'feature_order\']==[\'v\',\'t\',\'abs(v-t)\',\'v*t\'] and meta[\'frozen_encoder\'], \'特征定义不符\')\n        with np.load(p, allow_pickle=False) as z: d = {k:z[k].copy() for k in [\'x\',\'y\',\'ids\',\'image_ids\']}\n        x = d[\'x\']; v,t = x[:,:512],x[:,512:1024]\n        check(x.shape==(n,2048) and x.dtype==np.float32 and np.isfinite(x).all(), \'特征形状/精度/有限性异常\')\n        check(d[\'ids\'].tolist()==sid and d[\'image_ids\'].astype(str).tolist()==ids and d[\'y\'].tolist()==[r[\'label\'] for r in rr], \'缓存顺序或标签不符\')\n        check(np.allclose(np.linalg.norm(v,axis=1),1,atol=1e-4) and np.allclose(np.linalg.norm(t,axis=1),1,atol=1e-4), \'特征未归一化\')\n        check(np.allclose(x[:,1024:1536],abs(v-t),atol=1e-6) and np.allclose(x[:,1536:],v*t,atol=1e-6), \'特征分段不符\')\n        image_audit = old/\'handoff/manifests/image_audit.jsonl\'\n        check(record(image_audit)==meta[\'image_audit_sha256\'], \'图片审计来源变化\')\n        d[\'rows\']=rr; data[split]=d\n    runpath = old/\'handoff/predictions/train/run.json\'; run = read(runpath); record(runpath)\n    check(run[\'saved_order\']==[\'E\',\'N\',\'C\'] and run[\'manifest_sha256\']==files[\'handoff/manifests/train.jsonl\'], \'Teacher 顺序/来源不符\')\n    logits = {}\n    image_rows = rows(old/\'handoff/manifests/image_audit.jsonl\')\n    image_map = {r[\'sample_id\']:r for r in image_rows}\n    check(len(image_map)==len(image_rows), \'图片审计重复 ID\')\n    for method in [\'jev\',\'base\']:\n        p = old/f\'handoff/predictions/train/{method}.jsonl\'; rr=rows(p); digest=record(p)\n        if method==\'jev\': check(digest==lock[\'teacher_train_sha256\'], \'Jev 缓存与训练锁不符\')\n        byid={r[\'sample_id\']:r for r in rr}\n        check(len(byid)==len(rr)==6000 and set(byid)==set(data[\'train\'][\'ids\']), \'教师缓存缺失/重复/多余样本\')\n        ordered=[byid[r[\'sample_id\']] for r in data[\'train\'][\'rows\']]\n        for r, pred in zip(data[\'train\'][\'rows\'],ordered):\n            check(pred[\'model\']==method and all(pred[k]==r[k] for k in [\'label\',\'image_id\',\'image_file\',\'hypothesis\',\'split\']), \'教师缓存内容错配\')\n            check(pred[\'image_sha256\']==image_map[r[\'sample_id\']][\'file_sha256\'], \'教师图片哈希错配\')\n        z=np.asarray([r[\'logits\'] for r in ordered],dtype=np.float64)\n        check(z.shape==(6000,3) and np.isfinite(z).all(), \'教师 logits 异常\'); logits[method]=z\n    calpath=old/\'handoff/results/calibration.json\'; cal=read(calpath)\n    check(record(calpath)==lock[\'calibration_sha256\'] and cal[\'labels\']==[\'E\',\'N\',\'C\'], \'校准文件不符\')\n    params=cal[\'models\'][\'jev\'][\'selected_params\']\n    check(np.isfinite([params[\'T\'],params[\'b\']]).all() and params[\'T\']>0, \'校准参数异常\')\n    package_manifest=old.parent/\'文件校验清单.json\'\n    if package_manifest.exists():\n        hashes=read(package_manifest)\n        for name,digest in files.items(): check(hashes[\'reused_v1/\'+name]==digest, \'成果包总清单不符：\'+name)\n    else:\n        raise ValueError(\'需要10月13日成果包的文件校验清单.json，以验证 base 缓存来源。请指定完整成果包。\')\n    report=dict(status=\'PASS\', source=str(old), input_sha256=files, clip_revision=REV,\n                counts={s:len(d[\'y\']) for s,d in data.items()}, label_order=[\'E\',\'N\',\'C\'],\n                scope=\'历史缓存完整性与来源；未重新读取原图或重新推理 CLIP/Teacher\', calibration=params)\n    return dict(old=old, data=data, logits=logits, params=params, report=report)\n\ndef seed_all(seed):\n    import torch\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)\n\ndef make_head(method):\n    import torch.nn as nn\n    d,h=(512,2048) if method in [\'CE_image\',\'CE_text\'] else (2048,512)\n    return nn.Sequential(nn.Linear(d,h),nn.ReLU(),nn.Dropout(.1),nn.Linear(h,3))\n\ndef select_features(x, method):\n    return x[:,:512] if method==\'CE_image\' else x[:,512:1024] if method==\'CE_text\' else x\n\ndef target(ctx, method, tau):\n    if method in [\'CE\',\'CE_image\',\'CE_text\']: return None\n    z=ctx[\'logits\'][\'base\' if method==\'KD_base\' else \'jev\'].copy()\n    if method==\'KD_hard\': return z.argmax(1)\n    if method==\'KD_calibrated\': z=(z+np.array([ctx[\'params\'][\'b\'],0,0]))/ctx[\'params\'][\'T\']\n    z=z/tau; z-=z.max(1,keepdims=True); p=np.exp(z); return (p/p.sum(1,keepdims=True)).astype(np.float32)\n\ndef loss_fn(z,y,q,method,alpha,tau):\n    import torch.nn.functional as F\n    ce=F.cross_entropy(z,y)\n    if q is None: return ce\n    kd=F.cross_entropy(z,q) if method==\'KD_hard\' else tau*tau*F.kl_div(F.log_softmax(z/tau,dim=-1),q,reduction=\'batchmean\')\n    return (1-alpha)*ce+alpha*kd\n\ndef benchmark(ctx, out, methods=METHODS, seed=20261010, alpha=.75, tau=2., batch=128, warmup=10, steps=50, repeats=3):\n    import torch\n    check(torch.cuda.is_available(), \'短跑要求 GPU；不自动用 CPU 替代计时。\')\n    check(0<=alpha<=1 and tau>0 and batch>0 and warmup>=1 and steps>=1 and repeats>=1, \'计时参数不合法\')\n    check(len(methods)>0 and len(set(methods))==len(methods) and set(methods)<=set(METHODS), \'未知或重复方法\')\n    out=Path(out); out.mkdir(parents=True, exist_ok=False)\n    torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True\n    torch.backends.cuda.matmul.allow_tf32=False; torch.backends.cudnn.allow_tf32=True\n    env=dict(torch=str(torch.__version__), numpy=np.__version__, cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0),\n             python=platform.python_version(), cudnn=torch.backends.cudnn.version(), dtype=\'float32\', tf32_matmul=False, tf32_cudnn=True)\n    config=dict(methods=list(methods), seed=seed, alpha=alpha,tau=tau,batch=batch,warmup=warmup,steps=steps,repeats=repeats,\n                scope=\'GPU-resident cached features -> head training step; forward/loss/backward/AdamW; synchronized per step\',\n                excludes=[\'CLIP/Teacher inference\',\'disk read\',\'host-to-device copy\',\'validation\',\'checkpoint I/O\'], runtime_sha256=sha(__file__))\n    save(out/\'protocol.json\',dict(config=config,environment=env,cache_audit=ctx[\'report\']))\n    x=torch.tensor(ctx[\'data\'][\'train\'][\'x\'],device=\'cuda\'); y=torch.tensor(ctx[\'data\'][\'train\'][\'y\'],device=\'cuda\',dtype=torch.long)\n    check(batch<=len(y), \'batch 超过样本数\')\n    # Exact same sequence of full batches for every method and repeat; no tail batch in step timing.\n    g=torch.Generator(device=\'cpu\').manual_seed(seed)\n    batches=[]\n    while len(batches)<warmup+steps:\n        batches.extend(ix for ix in torch.randperm(len(y),generator=g).split(batch) if len(ix)==batch)\n    batches=[ix.to(\'cuda\') for ix in batches[:warmup+steps]]\n    raw=[]; summary=[]\n    for method in methods:\n        qn=target(ctx,method,tau); q=None if qn is None else torch.tensor(qn,device=\'cuda\',dtype=torch.long if method==\'KD_hard\' else torch.float32)\n        xx=select_features(x,method); times=[]; peaks=[]\n        for repeat in range(repeats):\n            seed_all(seed); model=make_head(method).cuda().train(); opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.0001)\n            def step(ix):\n                opt.zero_grad(set_to_none=True)\n                loss=loss_fn(model(xx[ix]),y[ix],None if q is None else q[ix],method,alpha,tau)\n                loss.backward(); opt.step(); return loss\n            for ix in batches[:warmup]: step(ix)\n            torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()\n            for j,ix in enumerate(batches[warmup:]):\n                torch.cuda.synchronize(); start=time.perf_counter(); loss=step(ix); torch.cuda.synchronize()\n                ms=(time.perf_counter()-start)*1000\n                value=float(loss.detach().cpu()); check(math.isfinite(value), \'短跑损失非有限\')\n                times.append(ms); raw.append(dict(method=method,repeat=repeat+1,step=j+1,ms=ms,loss=value))\n            peaks.append(torch.cuda.max_memory_allocated()/1024**2)\n            del opt,model; torch.cuda.empty_cache()\n        mean=float(np.mean(times)); median=float(np.median(times))\n        summary.append(dict(method=method,mean_step_ms=mean,median_step_ms=median,p95_step_ms=float(np.percentile(times,95)),\n                            samples_per_second=batch*1000/mean,peak_allocated_MiB=max(peaks),\n                            estimated_30_epochs_3_seeds_train_only_seconds=mean/1000*math.ceil(len(y)/batch)*30*3))\n        save(out/\'partial_results.json\',dict(summary=summary,raw=raw)); print(method, summary[-1],flush=True)\n        del q\n    save(out/\'results.json\',dict(summary=summary,raw=raw,note=\'工程短跑；全预算仅线性粗估，末批按整批估算；不含验证/保存，不能作为正式训练耗时或端到端加速比。\'))\n    with (out/\'timing_summary.csv\').open(\'w\',encoding=\'utf-8-sig\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=list(summary[0])); w.writeheader(); w.writerows(summary)\n    return summary\n', 'experiment.py': '"""Paired CE/label-smoothing/KD experiment and student temperature scaling."""\nfrom pathlib import Path\nfrom contextlib import contextmanager\nimport csv, io, json, math, os, platform, time, uuid\nimport numpy as np\nimport cache_audit as base\n\ncheck, sha, read, rows, save = base.check, base.sha, base.read, base.rows, base.save\nDEFAULT = dict(seeds=[20261010,20261011,20261012], methods=[\'CE\',\'LS\',\'KD_raw\'],\n               label_smoothing=.1, alpha=.75, tau=2., epochs=30, batch=128,\n               lr=.001, weight_decay=.0001, temperature_min=.05, temperature_max=20., temperature_points=401)\n\ndef atomic_bytes(p, value):\n    p=Path(p); p.parent.mkdir(parents=True,exist_ok=True)\n    tmp=p.with_name(p.name+\'.\'+uuid.uuid4().hex+\'.partial\')\n    with tmp.open(\'xb\') as f: f.write(value); f.flush(); os.fsync(f.fileno())\n    tmp.replace(p)\n\ndef save_torch(p, value):\n    import torch\n    b=io.BytesIO(); torch.save(value,b); atomic_bytes(p,b.getvalue())\n\ndef freeze(p,value):\n    if Path(p).exists(): check(read(p)==value,\'锁定内容发生变化，请使用新 WORK：\'+str(p))\n    else: save(p,value)\n\n@contextmanager\ndef exclusive(work):\n    p=Path(work)/\'ACTIVE.lock\'\n    with p.open(\'x\',encoding=\'utf-8\') as f: json.dump(dict(pid=os.getpid(),note=\'确认旧进程结束后才能手动移除此锁\'),f)\n    try: yield\n    finally: p.unlink()\n\ndef environment(device=\'cuda\'):\n    import torch\n    check(device in [\'cuda\',\'cpu\'],\'device 不合法\')\n    if device==\'cuda\': check(torch.cuda.is_available(),\'请启用 Colab GPU\')\n    torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True\n    torch.backends.cuda.matmul.allow_tf32=False; torch.backends.cudnn.allow_tf32=False\n    return dict(torch=str(torch.__version__),numpy=np.__version__,python=platform.python_version(),\n                device=device,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0) if device==\'cuda\' else None,\n                cudnn=torch.backends.cudnn.version(),tf32=False,dtype=\'float32\')\n\ndef prepare(source,work,config=None,device=\'cuda\'):\n    config=json.loads(json.dumps(DEFAULT if config is None else config))\n    check(set(config)==set(DEFAULT),\'配置项不完整\')\n    check(config[\'methods\']==[\'CE\',\'LS\',\'KD_raw\'],\'本对照固定 CE/LS/KD_raw 三组\')\n    check(len(config[\'seeds\'])==3 and len(set(config[\'seeds\']))==3,\'需要三个不同种子\')\n    check(0<config[\'label_smoothing\']<1 and 0<=config[\'alpha\']<=1 and config[\'tau\']>0,\'损失参数异常\')\n    check(config[\'epochs\']>=1 and config[\'batch\']>=1 and config[\'lr\']>0 and config[\'weight_decay\']>=0,\'训练参数异常\')\n    check(0<config[\'temperature_min\']<1<config[\'temperature_max\'] and config[\'temperature_points\']>=3,\'温度网格异常\')\n    source,work=Path(source).resolve(),Path(work).resolve()\n    check(not work.is_relative_to(source) and not source.is_relative_to(work),\'SOURCE/WORK 必须相互独立\')\n    ctx=base.audit(source); work.mkdir(parents=True,exist_ok=True)\n    ctx.update(work=work,config=config,device=device)\n    protocol=dict(kind=\'CE_LS_student_temperature_v1\',config=config,environment=environment(device),\n                  cache=ctx[\'report\'],code={n:sha(Path(__file__).parent/n) for n in [\'experiment.py\',\'cache_audit.py\']},\n                  selection=\'val_select max macro-F1; tie min uncalibrated NLL; tie earliest epoch\',\n                  calibration=\'per checkpoint scalar positive temperature, val_fit NLL grid minimum; no bias\',\n                  evaluation=\'val_select descriptive reused validation; optional historical test already exposed\')\n    freeze(work/\'protocol.json\',protocol); ctx[\'protocol_sha256\']=sha(work/\'protocol.json\')\n    return ctx\n\ndef softmax(z):\n    z=np.asarray(z,dtype=np.float64); z=z-z.max(1,keepdims=True); e=np.exp(z); return e/e.sum(1,keepdims=True)\n\ndef nll(z,y):\n    z=np.asarray(z,dtype=np.float64); m=z.max(1)\n    return float(np.mean(m+np.log(np.exp(z-m[:,None]).sum(1))-z[np.arange(len(y)),y]))\n\ndef metrics(z,y):\n    z=np.asarray(z,dtype=np.float64); y=np.asarray(y,dtype=np.int64)\n    check(z.shape==(len(y),3) and len(y)>0 and np.isfinite(z).all(),\'预测异常\')\n    check(np.isin(y,[0,1,2]).all(),\'标签异常\')\n    p=softmax(z); pred=z.argmax(1); cm=np.zeros((3,3),dtype=np.int64); np.add.at(cm,(y,pred),1)\n    f=2*cm.diagonal()/np.maximum(cm.sum(0)+cm.sum(1),1)\n    confidence=p.max(1); correct=pred==y; bins=np.minimum((confidence*10).astype(int),9)\n    ece=sum(float((bins==b).mean())*abs(float(correct[bins==b].mean())-float(confidence[bins==b].mean())) for b in range(10) if np.any(bins==b))\n    return dict(accuracy=float(correct.mean()),macro_f1=float(f.mean()),nll=nll(z,y),\n                brier=float(np.mean(np.sum((p-np.eye(3)[y])**2,axis=1))),ece10=ece,\n                f1_E=float(f[0]),f1_N=float(f[1]),f1_C=float(f[2]))\n\ndef fit_temperature(z,y,cfg):\n    grid=np.unique(np.r_[1.,np.geomspace(cfg[\'temperature_min\'],cfg[\'temperature_max\'],cfg[\'temperature_points\'])])\n    values=[nll(z/t,y) for t in grid]\n    i=min(range(len(grid)),key=lambda i:(values[i],abs(math.log(grid[i]))))\n    return dict(T=float(grid[i]),fit_nll_raw=nll(z,y),fit_nll_calibrated=values[i],\n                at_boundary=i in [0,len(grid)-1],grid=grid.tolist(),nll_grid=values)\n\ndef checked_feature(ctx,split,folder):\n    folder=Path(folder); p=folder/f\'{split}.npz\'; meta=read(folder/f\'{split}_meta.json\')\n    manifest=ctx[\'old\']/f\'handoff/manifests/{split}.jsonl\'; rr=rows(manifest)\n    check(meta[\'npz_sha256\']==sha(p) and meta[\'manifest_sha256\']==sha(manifest),\'新增特征哈希不符\')\n    check(meta[\'revision\']==base.REV and meta[\'frozen_encoder\'] and meta[\'feature_order\']==[\'v\',\'t\',\'abs(v-t)\',\'v*t\'],\'新增特征定义不符\')\n    check(meta[\'image_audit_sha256\']==sha(ctx[\'old\']/\'handoff/manifests/image_audit.jsonl\'),\'图像审计来源变化\')\n    with np.load(p,allow_pickle=False) as a: d={k:a[k].copy() for k in [\'x\',\'y\',\'ids\',\'image_ids\']}\n    x=d[\'x\']; check(x.shape==(len(rr),2048) and x.dtype==np.float32 and np.isfinite(x).all(),\'新增特征异常\')\n    check(d[\'ids\'].tolist()==[r[\'sample_id\'] for r in rr] and d[\'y\'].tolist()==[r[\'label\'] for r in rr] and d[\'image_ids\'].astype(str).tolist()==[str(r[\'image_id\']) for r in rr],\'新增特征 ID/标签错配\')\n    v,t=x[:,:512],x[:,512:1024]\n    check(np.allclose(np.linalg.norm(v,axis=1),1,atol=1e-4) and np.allclose(np.linalg.norm(t,axis=1),1,atol=1e-4),\'新增特征未归一化\')\n    check(np.allclose(x[:,1024:1536],abs(v-t),atol=1e-6) and np.allclose(x[:,1536:],v*t,atol=1e-6),\'新增特征分段不符\')\n    d[\'rows\']=rr; return d\n\ndef feature_data(ctx,split):\n    if split in ctx[\'data\']: return ctx[\'data\'][split]\n    return checked_feature(ctx,split,ctx[\'work\']/\'features\')\n\ndef extract_missing(ctx,images,split=\'val_fit\'):\n    """Only new output cache; reuse requires verified manifest, payload and provenance."""\n    import torch\n    import torch.nn.functional as F\n    import importlib.metadata as md\n    from PIL import Image\n    from transformers import CLIPModel,CLIPProcessor\n    check(split in [\'val_fit\',\'test\'],\'只补齐 val_fit 或 test\')\n    if split==\'test\': check((ctx[\'work\']/\'student_calibration.json\').exists(),\'先锁定校准再准备测试输入\')\n    folder=ctx[\'work\']/\'features\'; folder.mkdir(exist_ok=True)\n    p=folder/f\'{split}.npz\'; mp=folder/f\'{split}_meta.json\'\n    if p.exists() and mp.exists(): return checked_feature(ctx,split,folder)\n    check(ctx[\'device\']==\'cuda\',\'正式特征提取需要 GPU\')\n    check(md.version(\'transformers\')==\'4.57.1\',\'请使用 transformers 4.57.1\')\n    images=Path(images).resolve(); audit_path=ctx[\'old\']/\'handoff/manifests/image_audit.jsonl\'\n    audit={r[\'sample_id\']:r for r in rows(audit_path)}\n    manifest=ctx[\'old\']/f\'handoff/manifests/{split}.jsonl\'; rr=rows(manifest)\n    known=ctx[\'data\'][\'val_select\'][\'rows\'][:32]\n    def image_path(r):\n        path=(images/r[\'image_file\']).resolve()\n        check(path.is_relative_to(images),\'图片路径越界\')\n        check(path.is_file() and sha(path)==audit[r[\'sample_id\']][\'file_sha256\'],\'图片缺失/哈希不符：\'+str(path))\n        return path\n    for r in rr+known: image_path(r)\n    with exclusive(ctx[\'work\']):\n        model=CLIPModel.from_pretrained(\'openai/clip-vit-base-patch32\',revision=base.REV).to(\'cuda\',dtype=torch.float32).eval()\n        processor=CLIPProcessor.from_pretrained(\'openai/clip-vit-base-patch32\',revision=base.REV)\n        def encode(batch):\n            ims=[]\n            for r in batch:\n                with Image.open(image_path(r)) as im: ims.append(im.convert(\'RGB\'))\n            inp=processor(text=[r[\'hypothesis\'] for r in batch],images=ims,padding=True,truncation=True,max_length=77,return_tensors=\'pt\').to(\'cuda\')\n            with torch.inference_mode():\n                output=model(**inp); v=F.normalize(output.image_embeds.float(),dim=-1); t=F.normalize(output.text_embeds.float(),dim=-1)\n                return torch.cat([v,t,abs(v-t),v*t],dim=-1).cpu().numpy()\n        try:\n            delta=float(abs(encode(known)-ctx[\'data\'][\'val_select\'][\'x\'][:32]).max())\n            check(delta<=1e-4,f\'与旧特征兼容检查失败：max abs={delta}；不要放宽阈值后混用。\')\n            provenance=dict(environment=environment(ctx[\'device\']),transformers=md.version(\'transformers\'),revision=base.REV)\n            freeze(folder/\'extraction_environment.json\',provenance)\n            values=[]; truncated=[]\n            for start in range(0,len(rr),32):\n                batch=rr[start:start+32]; values.append(encode(batch))\n                truncated += [r[\'sample_id\'] for r in batch if len(processor.tokenizer(r[\'hypothesis\'])[\'input_ids\'])>77]\n            b=io.BytesIO(); np.savez_compressed(b,x=np.concatenate(values),y=[r[\'label\'] for r in rr],ids=[r[\'sample_id\'] for r in rr],image_ids=[r[\'image_id\'] for r in rr])\n            atomic_bytes(p,b.getvalue()); save(mp,dict(npz_sha256=sha(p),manifest_sha256=sha(manifest),revision=base.REV,\n                 frozen_encoder=True,feature_order=[\'v\',\'t\',\'abs(v-t)\',\'v*t\'],image_audit_sha256=sha(audit_path),\n                 compatibility_max_abs=delta,truncated_ids=truncated,extraction_environment=provenance))\n        finally:\n            del model; torch.cuda.empty_cache()\n    return checked_feature(ctx,split,folder)\n\ndef training_loss(z,y,q,method,cfg):\n    import torch.nn.functional as F\n    check(method in [\'Text_CE\',\'Text_KD_raw\'],\'文本实验不使用标签平滑或校准KD\')\n    ce=F.cross_entropy(z,y)\n    if method==\'Text_CE\': return ce\n    check(method==\'Text_KD_raw\',\'文本实验只允许原始 KD\')\n    return (1-cfg[\'alpha\'])*ce+cfg[\'alpha\']*cfg[\'tau\']**2*F.kl_div(F.log_softmax(z/cfg[\'tau\'],dim=-1),q,reduction=\'batchmean\')\n\ndef predict(model,x,device):\n    import torch\n    model.eval(); zs=[]\n    with torch.inference_mode():\n        for start in range(0,len(x),256): zs.append(model(torch.as_tensor(x[start:start+256],device=device)).cpu().numpy())\n    return np.concatenate(zs)\n\ndef train_one(ctx,method,seed,stop_after_epoch=None):\n    """Epoch-atomic resume. Epoch-specific RNG makes resumed dropout/shuffle identical."""\n    import torch\n    cfg=ctx[\'config\']; device=ctx[\'device\']; folder=ctx[\'work\']/f\'runs/{method}_{seed}\'; folder.mkdir(parents=True,exist_ok=True)\n    statepath=folder/\'state.pt\'; donepath=folder/\'done.json\'\n    identity=dict(protocol_sha256=ctx[\'protocol_sha256\'],method=method,seed=seed)\n    if donepath.exists():\n        done=read(donepath); check(done[\'identity\']==identity and done[\'checkpoint_sha256\']==sha(folder/\'best.pt\'),\'已完成 checkpoint 变化\')\n        return done\n    check(not (ctx[\'work\']/\'selection.json\').exists(),\'模型已锁定，不能重新训练\')\n    a=ctx[\'data\'][\'train\']; b=ctx[\'data\'][\'val_select\']; base.seed_all(seed)\n    check(method in [\'Text_CE\',\'Text_KD_raw\'],\'未知文本方法\')\n    check(a[\'x\'].shape[1]==b[\'x\'].shape[1]==512,\'文本输入必须512维\')\n    model=base.make_head(\'CE_text\').to(device); opt=torch.optim.AdamW(model.parameters(),lr=cfg[\'lr\'],weight_decay=cfg[\'weight_decay\'])\n    x=torch.as_tensor(a[\'x\'],device=device); y=torch.as_tensor(a[\'y\'],dtype=torch.long,device=device)\n    q=torch.tensor(base.target(ctx,\'KD_raw\',cfg[\'tau\']),dtype=torch.float32,device=device) if method==\'Text_KD_raw\' else None\n    history=[]; best=(-float(\'inf\'),-float(\'inf\')); best_state=None; best_epoch=None; start=1\n    if statepath.exists():\n        s=torch.load(statepath,map_location=\'cpu\',weights_only=True); check(s[\'identity\']==identity,\'断点来源不一致\')\n        model.load_state_dict(s[\'model\']); opt.load_state_dict(s[\'optimizer\']); history=s[\'history\']; best=tuple(s[\'best\'])\n        best_state=s[\'best_state\']; best_epoch=s[\'best_epoch\']; start=s[\'epoch\']+1\n    for epoch in range(start,cfg[\'epochs\']+1):\n        base.seed_all(seed+100000*epoch); model.train(); total=0.; begun=time.perf_counter()\n        g=torch.Generator(device=\'cpu\').manual_seed(seed+epoch)\n        for ix in torch.randperm(len(y),generator=g).split(cfg[\'batch\']):\n            ix=ix.to(device); opt.zero_grad(set_to_none=True)\n            loss=training_loss(model(x[ix]),y[ix],None if q is None else q[ix],method,cfg)\n            check(bool(torch.isfinite(loss).item()),\'训练损失非有限\')\n            loss.backward(); opt.step(); total+=float(loss.detach().cpu())*len(ix)\n        z=predict(model,b[\'x\'],device); m=metrics(z,b[\'y\']); key=(m[\'macro_f1\'],-m[\'nll\'])\n        if key>best:\n            best=key; best_epoch=epoch; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}\n        history.append(dict(epoch=epoch,loss=total/len(y),validation=m,train_and_validation_seconds=time.perf_counter()-begun))\n        save_torch(statepath,dict(identity=identity,epoch=epoch,model=model.state_dict(),optimizer=opt.state_dict(),history=history,\n                                  best=list(best),best_state=best_state,best_epoch=best_epoch))\n        print(method,seed,f\'{epoch}/{cfg["epochs"]}\',f\'val F1={m["macro_f1"]:.4f}\',flush=True)\n        if stop_after_epoch is not None and epoch>=stop_after_epoch and epoch<cfg[\'epochs\']: return None\n    check(best_state is not None and len(history)==cfg[\'epochs\'],\'训练不完整\')\n    save_torch(folder/\'best.pt\',best_state); save(folder/\'history.json\',history)\n    done=dict(identity=identity,method=method,seed=seed,best_epoch=best_epoch,validation_macro_f1=best[0],validation_nll=-best[1],\n              checkpoint=str((folder/\'best.pt\').relative_to(ctx[\'work\'])),checkpoint_sha256=sha(folder/\'best.pt\'),history_sha256=sha(folder/\'history.json\'))\n    save(donepath,done); return done\n\ndef train_all(ctx):\n    environment(ctx[\'device\']); selected=[]\n    with exclusive(ctx[\'work\']):\n        for seed in ctx[\'config\'][\'seeds\']:\n            for method in ctx[\'config\'][\'methods\']: selected.append(train_one(ctx,method,seed))\n        freeze(ctx[\'work\']/\'selection.json\',dict(protocol_sha256=ctx[\'protocol_sha256\'],models=selected))\n    return selected\n\ndef selected_models(ctx):\n    selection=read(ctx[\'work\']/\'selection.json\'); check(selection[\'protocol_sha256\']==ctx[\'protocol_sha256\'],\'选模协议变化\')\n    expected={(m,s) for m in ctx[\'config\'][\'methods\'] for s in ctx[\'config\'][\'seeds\']}\n    check(len(selection[\'models\'])==len(expected) and {(m[\'method\'],m[\'seed\']) for m in selection[\'models\']}==expected,\'选模不完整\')\n    for m in selection[\'models\']: check(sha(ctx[\'work\']/m[\'checkpoint\'])==m[\'checkpoint_sha256\'],\'checkpoint 损坏\')\n    return selection[\'models\']\n\ndef model_logits(ctx,model_info,data):\n    import torch\n    check(model_info[\'method\'] in [\'Text_CE\',\'Text_KD_raw\'],\'只能读取文本模型\')\n    check(data[\'x\'].shape[1]==512,\'文本输入必须512维\')\n    model=base.make_head(\'CE_text\').to(ctx[\'device\'])\n    model.load_state_dict(torch.load(ctx[\'work\']/model_info[\'checkpoint\'],map_location=\'cpu\',weights_only=True))\n    return predict(model,data[\'x\'],ctx[\'device\'])\n\ndef calibrate(ctx):\n    data=feature_data(ctx,\'val_fit\'); models=selected_models(ctx); path=ctx[\'work\']/\'student_calibration.json\'\n    inputs=dict(protocol_sha256=ctx[\'protocol_sha256\'],selection_sha256=sha(ctx[\'work\']/\'selection.json\'),\n                val_fit_npz_sha256=sha(ctx[\'work\']/\'features/val_fit.npz\'),val_fit_meta_sha256=sha(ctx[\'work\']/\'features/val_fit_meta.json\'))\n    if path.exists():\n        result=read(path); check(result[\'inputs\']==inputs,\'校准输入变化\'); return result\n    with exclusive(ctx[\'work\']):\n        result=[]\n        for m in models:\n            z=model_logits(ctx,m,data); fit=fit_temperature(z,data[\'y\'],ctx[\'config\'])\n            check(np.array_equal(z.argmax(1),(z/fit[\'T\']).argmax(1)),\'正温度不应改变 argmax\')\n            result.append(dict(method=m[\'method\'],seed=m[\'seed\'],checkpoint_sha256=m[\'checkpoint_sha256\'],**fit))\n            print(\'temperature\',m[\'method\'],m[\'seed\'],fit[\'T\'],\'boundary=\',fit[\'at_boundary\'],flush=True)\n        value=dict(inputs=inputs,models=result,scope=\'fit NLL on val_fit; this fit loss is not evaluation performance\')\n        freeze(path,value)\n    return value\n\ndef csvwrite(p,records):\n    if not records: return\n    b=io.StringIO(newline=\'\'); w=csv.DictWriter(b,fieldnames=list(records[0])); w.writeheader(); w.writerows(records)\n    atomic_bytes(p,b.getvalue().encode(\'utf-8-sig\'))\n\ndef report(ctx,split=\'val_select\',allow_historical_test=False):\n    check(split in [\'val_select\',\'test\'],\'报告划分不合法\')\n    check(split!=\'test\' or allow_historical_test,\'历史测试复评开关未开启\')\n    models=selected_models(ctx); calibration=calibrate(ctx); data=feature_data(ctx,split)\n    out=ctx[\'work\']/f\'reports/{split}\'; out.mkdir(parents=True,exist_ok=True)\n    inputs=dict(protocol_sha256=ctx[\'protocol_sha256\'],selection_sha256=sha(ctx[\'work\']/\'selection.json\'),calibration_sha256=sha(ctx[\'work\']/\'student_calibration.json\'),\n                feature_sha256=sha((ctx[\'old\'] if split==\'val_select\' else ctx[\'work\'])/f\'features/{split}.npz\'))\n    if (out/\'COMPLETE.json\').exists():\n        done=read(out/\'COMPLETE.json\'); check(done[\'inputs\']==inputs,\'已完成报告输入改变\')\n        for name,h in done[\'files\'].items(): check(sha(out/name)==h,\'报告文件损坏\')\n        return out\n    with exclusive(ctx[\'work\']):\n        values=[]; predictions=[]\n        for m in models:\n            fit=next(c for c in calibration[\'models\'] if c[\'method\']==m[\'method\'] and c[\'seed\']==m[\'seed\'])\n            check(fit[\'checkpoint_sha256\']==m[\'checkpoint_sha256\'],\'校准与模型错配\')\n            z=model_logits(ctx,m,data)\n            for variant,T in [(\'raw\',1.),(\'temperature\',fit[\'T\'])]:\n                zz=z.astype(np.float64)/T; p=softmax(zz)\n                values.append(dict(method=m[\'method\'],seed=m[\'seed\'],variant=variant,T=T,**metrics(zz,data[\'y\'])))\n                for i,sid in enumerate(data[\'ids\']):\n                    predictions.append(dict(sample_id=str(sid),image_id=str(data[\'image_ids\'][i]),label=int(data[\'y\'][i]),method=m[\'method\'],seed=m[\'seed\'],variant=variant,\n                                            logit_E=float(zz[i,0]),logit_N=float(zz[i,1]),logit_C=float(zz[i,2]),p_E=float(p[i,0]),p_N=float(p[i,1]),p_C=float(p[i,2]),prediction=int(zz[i].argmax())))\n            a,b=values[-2:]; check(a[\'accuracy\']==b[\'accuracy\'] and a[\'macro_f1\']==b[\'macro_f1\'],\'温度校准意外改变分类\')\n        metric_names=[\'accuracy\',\'macro_f1\',\'nll\',\'brier\',\'ece10\',\'f1_E\',\'f1_N\',\'f1_C\']; summary=[]; deltas=[]\n        for method in ctx[\'config\'][\'methods\']:\n            for variant in [\'raw\',\'temperature\']:\n                group=[r for r in values if r[\'method\']==method and r[\'variant\']==variant]\n                summary.append(dict(method=method,variant=variant,**{k+suffix:float(fn([r[k] for r in group])) for k in metric_names for suffix,fn in [(\'_mean\',np.mean),(\'_seed_sd\',lambda a:np.std(a,ddof=1))]}))\n        def get(method,seed,variant): return next(r for r in values if r[\'method\']==method and r[\'seed\']==seed and r[\'variant\']==variant)\n        for seed in ctx[\'config\'][\'seeds\']:\n            for variant in [\'raw\',\'temperature\']:\n                for a,b in [(\'LS\',\'CE\'),(\'KD_raw\',\'CE\'),(\'KD_raw\',\'LS\')]:\n                    ra,rb=get(a,seed,variant),get(b,seed,variant)\n                    deltas.append(dict(comparison=a+\' minus \'+b,seed=seed,variant=variant,**{k:ra[k]-rb[k] for k in metric_names}))\n            for method in ctx[\'config\'][\'methods\']:\n                ra,rb=get(method,seed,\'temperature\'),get(method,seed,\'raw\')\n                deltas.append(dict(comparison=method+\' temperature minus raw\',seed=seed,variant=\'postcalibration\',**{k:ra[k]-rb[k] for k in metric_names}))\n        csvwrite(out/\'per_seed.csv\',values); csvwrite(out/\'summary.csv\',summary); csvwrite(out/\'paired_deltas.csv\',deltas); csvwrite(out/\'predictions.csv\',predictions)\n        save(out/\'metrics.json\',dict(inputs=inputs,per_seed=values,summary=summary,paired_deltas=deltas))\n        note=(\'验证选择集，已用于 checkpoint 选择，仅作开发性描述。\' if split==\'val_select\' else \'历史测试集已被查看；本轮复评不能称为全新独立确认。\')\n        text=\'# CE / 标签平滑 / 学生后校准\\n\\n\'+note+\'\\n\\n所有准确率、F1、ECE 用 0–1 原值；差值也是原值。seed_sd 为三个种子之间的样本标准差，不是置信区间。NLL/Brier/ECE 越低越好。\\n\\n温度只改变概率，不改变分类；不保证评价集的概率指标一定改善。边界温度见 student_calibration.json，不据测试结果扩网格。\\n\\n|方法|版本|Macro-F1 均值|NLL 均值|Brier 均值|ECE 均值|\\n|---|---|---:|---:|---:|---:|\\n\'\n        for r in summary: text+=f"|{r[\'method\']}|{r[\'variant\']}|{r[\'macro_f1_mean\']:.4f}|{r[\'nll_mean\']:.4f}|{r[\'brier_mean\']:.4f}|{r[\'ece10_mean\']:.4f}|\\n"\n        atomic_bytes(out/\'README.md\',text.encode(\'utf-8\'))\n        names=[\'per_seed.csv\',\'summary.csv\',\'paired_deltas.csv\',\'predictions.csv\',\'metrics.json\',\'README.md\']\n        save(out/\'COMPLETE.json\',dict(inputs=inputs,files={n:sha(out/n) for n in names}))\n    return out\n', 'text_only.py': '"""Six text-only runs with read-only Stage2 multimodal baselines."""\nfrom pathlib import Path, PurePosixPath\nimport csv,io,json,hashlib,zipfile,uuid,copy\nimport numpy as np\nimport experiment as e\n\nSEEDS=[20261010,20261011,20261012]\nMETHODS=[\'Text_CE\',\'Text_KD_raw\']\nMETRICS=[\'macro_f1\',\'accuracy\',\'f1_E\',\'f1_N\',\'f1_C\',\'nll\',\'ece10\',\'brier\']\n\ndef table(path,records,fields=None):\n    fields=fields or list(dict.fromkeys(k for r in records for k in r)) or [\'status\']\n    out=io.StringIO(newline=\'\'); w=csv.DictWriter(out,fieldnames=fields); w.writeheader(); w.writerows(records)\n    e.atomic_bytes(path,out.getvalue().encode(\'utf-8-sig\'))\n\ndef csvrows(payload): return list(csv.DictReader(io.StringIO(payload.decode(\'utf-8-sig\'))))\n\ndef text_slice(x):\n    e.check(x.ndim==2 and x.shape[1]==2048,\'缓存须为2048维图文特征\')\n    return np.ascontiguousarray(x[:,512:1024],dtype=np.float32)\n\ndef head_info():\n    import torch.nn as nn\n    text=e.base.make_head(\'CE_text\'); multi=e.base.make_head(\'CE\')\n    def describe(model):\n        linear=[[m.in_features,m.out_features] for m in model if isinstance(m,nn.Linear)]\n        return dict(linear_shapes=linear,parameters=sum(p.numel() for p in model.parameters()))\n    a,b=describe(text),describe(multi)\n    e.check(a[\'linear_shapes\']==[[512,2048],[2048,3]] and b[\'linear_shapes\']==[[2048,512],[512,3]],\'分类头与既定方案不同，停止\')\n    return dict(text=a,multimodal=b,difference=a[\'parameters\']-b[\'parameters\'],difference_percent=100*(a[\'parameters\']-b[\'parameters\'])/b[\'parameters\'])\n\nclass Package:\n    def __init__(self,path):\n        self.path=Path(path).resolve(); self.z=None\n        if self.path.is_file():\n            self.z=zipfile.ZipFile(self.path); names=self.z.namelist()\n            e.check(len(names)==len(set(names)),\'ZIP 成员重复\')\n            matches=[n for n in names if n==\'PACKAGE_SHA256.json\' or n.endswith(\'/PACKAGE_SHA256.json\')]\n            e.check(len(matches)==1,\'需要 Stage2 正式结果ZIP，含PACKAGE_SHA256.json\')\n            self.prefix=matches[0][:-len(\'PACKAGE_SHA256.json\')]\n        else:\n            e.check((self.path/\'PACKAGE_SHA256.json\').is_file(),\'解压目录应直接包含PACKAGE_SHA256.json\')\n        self.hashes=json.loads(self.read(\'PACKAGE_SHA256.json\',verify=False))\n    def read(self,name,verify=True):\n        parts=PurePosixPath(name).parts\n        e.check(\'..\' not in parts and \'\\\\\' not in name and \':\' not in name and not name.startswith(\'/\'),\'非法包内路径\')\n        data=self.z.read(self.prefix+name) if self.z else (self.path/name).read_bytes()\n        if verify: e.check(name in self.hashes and hashlib.sha256(data).hexdigest()==self.hashes[name],\'Stage2文件哈希错误：\'+name)\n        return data\n    def json(self,name): return json.loads(self.read(name))\n    def close(self):\n        if self.z: self.z.close()\n\ndef baseline_from_package(ctx,path,work):\n    pkg=Package(path); kept={}\n    def read(name):\n        b=pkg.read(name); kept[name]=b; return b\n    def js(name): return json.loads(read(name))\n    try:\n        protocol=js(\'protocol.json\'); status=js(\'report/status.json\')\n        e.check(protocol[\'kind\']==\'Stage2_Hazel_18KD_v1\' and status[\'complete\'] and status[\'completed_conditions\']==18,\'Stage2 18条件未完成\')\n        e.check(protocol[\'cache\'][\'input_sha256\']==ctx[\'report\'][\'input_sha256\'],\'Stage2与本轮缓存不一致\')\n        cfg=copy.deepcopy(protocol[\'config\'])\n        fixed=dict(seeds=SEEDS,lr=.0003,tau=2.,epochs=30,batch=128,weight_decay=.0001)\n        for k,v in fixed.items(): e.check(cfg[k]==v,\'Stage2配置不符：\'+k)\n        group=\'groups/KD_raw_alpha0.75\'\n        gp=js(group+\'/protocol.json\'); selection=js(group+\'/selection.json\')\n        e.check(gp[\'parent_sha256\']==hashlib.sha256(kept[\'protocol.json\']).hexdigest(),\'KD上级协议不符\')\n        e.check(gp[\'method\']==\'KD_raw\' and gp[\'config\'][\'alpha\']==.75 and gp[\'teacher_params\']==dict(T=1.,b=0.),\'基线必须是alpha=.75原始KD\')\n        e.check(selection[\'protocol_sha256\']==hashlib.sha256(kept[group+\'/protocol.json\']).hexdigest(),\'KD选模协议不符\')\n        e.check(len(selection[\'models\'])==3 and {m[\'seed\'] for m in selection[\'models\']}==set(SEEDS),\'图文KD必须三个种子\')\n        ds=js(\'Day2_preserved/supervised_selection.json\'); dp=js(\'Day2_preserved/protocol.json\')\n        e.check(ds[\'parent_sha256\']==hashlib.sha256(kept[\'Day2_preserved/protocol.json\']).hexdigest(),\'CE选模协议不符\')\n        e.check(dp[\'input_sha256\']==ctx[\'report\'][\'input_sha256\'],\'CE输入缓存不一致\')\n        ce=[m for m in ds[\'models\'] if m[\'method\']==\'CE\' and m[\'lr\']==.0003]\n        e.check(len(ce)==3 and {m[\'seed\'] for m in ce}==set(SEEDS),\'图文CE必须同学习率三个种子\')\n        day2_preds=csvrows(read(\'Day2_preserved/report/predictions.csv\'))\n        summary=csvrows(read(\'report/combined_Day2_KD_per_seed.csv\'))\n        ids=ctx[\'data\'][\'val_select\'][\'ids\'].tolist(); labels=ctx[\'data\'][\'val_select\'][\'y\']; baseline=[]\n        for method,models,root in [(\'MM_CE\',ce,\'Day2_preserved\'),(\'MM_KD_raw\',selection[\'models\'],group)]:\n            for m in models:\n                cp=root+\'/\'+m[\'checkpoint\'].replace(\'\\\\\',\'/\'); folder=cp.rsplit(\'/\',1)[0]\n                e.check(hashlib.sha256(read(cp)).hexdigest()==m[\'checkpoint_sha256\'],\'基线checkpoint不符\')\n                hist=read(folder+\'/history.json\'); e.check(hashlib.sha256(hist).hexdigest()==m[\'history_sha256\'],\'基线history不符\')\n                h=json.loads(hist); best=max(h,key=lambda r:(r[\'validation\'][\'macro_f1\'],-r[\'validation\'][\'nll\'],-r[\'epoch\']))\n                e.check(len(h)==30 and best[\'epoch\']==m[\'best_epoch\'],\'基线预算/选模不符\')\n                read(folder+\'/done.json\')\n                if method==\'MM_KD_raw\': preds=csvrows(read(folder+\'/predictions.csv\'))\n                else: preds=[r for r in day2_preds if r[\'variant\']==\'raw\' and r[\'model\']==f"CE_eps0_{m[\'seed\']}"]\n                byid={r[\'sample_id\']:r for r in preds}\n                e.check(len(byid)==len(preds)==len(ids) and set(byid)==set(ids),\'基线预测样本缺失、重复或错配\')\n                ordered=[byid[sid] for sid in ids]\n                e.check([int(r[\'label\']) for r in ordered]==labels.tolist(),\'基线gold错配\')\n                z=np.array([[float(r[\'logit_\'+k]) for k in \'ENC\'] for r in ordered]); scores=e.metrics(z,labels)\n                ref=[r for r in summary if r[\'seed\']==str(m[\'seed\']) and r[\'variant\']==\'raw\' and ((method==\'MM_CE\' and r[\'method\']==\'CE\') or (method==\'MM_KD_raw\' and r[\'method\']==\'KD_raw\' and float(r[\'alpha\'])==.75))]\n                e.check(len(ref)==1 and all(np.isclose(scores[k],float(ref[0][k]),atol=1e-10,rtol=0) for k in METRICS),\'基线重算指标不一致\')\n                baseline.append(dict(method=method,seed=m[\'seed\'],source=\'Stage2_reused\',best_epoch=m[\'best_epoch\'],**scores))\n                table(work/\'baselines\'/f\'{method}_{m["seed"]}_predictions.csv\',[dict(sample_id=sid,label=int(labels[i]),prediction=int(z[i].argmax()),**{\'logit_\'+k:float(z[i,j]) for j,k in enumerate(\'ENC\')},**{\'p_\'+k:float(e.softmax(z)[i,j]) for j,k in enumerate(\'ENC\')}) for i,sid in enumerate(ids)])\n        for name,b in kept.items():\n            dest=work/\'Stage2_reference\'/name\n            if dest.exists(): e.check(dest.read_bytes()==b,\'已有基线快照变化\')\n            else: e.atomic_bytes(dest,b)\n        return protocol,cfg,baseline,{n:hashlib.sha256(b).hexdigest() for n,b in kept.items()}\n    finally: pkg.close()\n\ndef environment_check(expected,current):\n    diff={k:dict(previous=expected.get(k),current=current.get(k)) for k in set(expected)|set(current) if k not in expected or k not in current or expected[k]!=current[k]}\n    allowed={\'python\':dict(previous=\'3.13.15\',current=\'3.13.16\')}\n    e.check(not diff or diff==allowed,\'运行环境不一致：\'+json.dumps(diff,ensure_ascii=False))\n    return dict(previous=expected,current=current,differences=diff,policy=\'exact; documented Python 3.13.15 -> 3.13.16 exception only\')\n\ndef prepare(source,stage2_results,work,device=\'cuda\'):\n    source,stage2_results,work=map(lambda p:Path(p).resolve(),[source,stage2_results,work])\n    for p in [source,stage2_results]: e.check(not work.is_relative_to(p) and not p.is_relative_to(work),\'新WORK必须与所有输入独立\')\n    ctx=e.base.audit(source); work.mkdir(parents=True,exist_ok=True)\n    prior,cfg,baseline,hashes=baseline_from_package(ctx,stage2_results,work)\n    env=e.environment(device); env_diff=environment_check(prior[\'environment\'],env)\n    heads=head_info(); cfg.update(methods=METHODS,alpha=.75,lr=.0003,tau=2.,label_smoothing=0.)\n    # New arrays contain ONLY t. No v, abs(v-t), or v*t survives in student input.\n    ctx[\'data\']={split:dict(d,x=text_slice(d[\'x\'])) for split,d in ctx[\'data\'].items()}\n    q=e.base.target(ctx,\'KD_raw\',2.)\n    target_sha=hashlib.sha256(q.tobytes(order=\'C\')).hexdigest()\n    protocol=dict(kind=\'Stage2_Hazel_text_only_6_v1\',config=cfg,environment=env,environment_comparison=env_diff,cache=ctx[\'report\'],\n        source_stage2=str(stage2_results),baseline_files=hashes,heads=heads,\n        student_input=\'x[:,512:1024] contiguous float32; t only\',teacher_input=\'image+text; unchanged original Jev logits\',\n        target=dict(type=\'raw logits -> softmax(logits/2), applied once\',label_order=[\'E\',\'N\',\'C\'],sha256_float32_C=target_sha,shape=list(q.shape)),\n        selection=\'val_select max raw Macro-F1, min NLL, earliest epoch\',evaluation=\'raw student logits; 10 equal-width ECE bins; no test or postcalibration\',\n        scope=\'input contribution diagnostic; not proof of cross-modal reasoning; seed SD is not paired CI\',\n        code={n:e.sha(Path(__file__).parent/n) for n in [\'cache_audit.py\',\'experiment.py\',\'text_only.py\']})\n    e.freeze(work/\'protocol.json\',protocol)\n    ctx.update(work=work,config=cfg,device=device,baseline=baseline,protocol=protocol,protocol_sha256=e.sha(work/\'protocol.json\'))\n    return ctx\n\ndef export_prediction(ctx,m):\n    folder=(ctx[\'work\']/m[\'checkpoint\']).parent; rp=folder/\'result.json\'\n    identity=dict(protocol_sha256=ctx[\'protocol_sha256\'],checkpoint_sha256=m[\'checkpoint_sha256\'])\n    if rp.exists():\n        r=e.read(rp); e.check(r[\'identity\']==identity and r[\'prediction_sha256\']==e.sha(folder/\'predictions.csv\'),\'文本预测损坏\'); return r[\'metrics\']\n    d=ctx[\'data\'][\'val_select\']; z=e.model_logits(ctx,m,d); p=e.softmax(z)\n    row=dict(method=m[\'method\'],seed=m[\'seed\'],source=\'Text_new\',best_epoch=m[\'best_epoch\'],**e.metrics(z,d[\'y\']))\n    table(folder/\'predictions.csv\',[dict(sample_id=str(sid),gold=int(d[\'y\'][i]),prediction=int(z[i].argmax()),**{\'logit_\'+k:float(z[i,j]) for j,k in enumerate(\'ENC\')},**{\'p_\'+k:float(p[i,j]) for j,k in enumerate(\'ENC\')}) for i,sid in enumerate(d[\'ids\'])])\n    e.freeze(rp,dict(identity=identity,metrics=row,prediction_sha256=e.sha(folder/\'predictions.csv\')))\n    return row\n\ndef collect(ctx):\n    inventory=[]; records=[]; families={}\n    for method in METHODS:\n        rr=[]\n        for seed in SEEDS:\n            folder=ctx[\'work\']/f\'runs/{method}_{seed}\'; row=dict(method=method,seed=seed,status=\'missing\')\n            if (folder/\'state.pt\').exists(): row[\'status\']=\'in_progress\'\n            if (folder/\'done.json\').exists():\n                m=e.read(folder/\'done.json\')\n                e.check(m[\'identity\']==dict(protocol_sha256=ctx[\'protocol_sha256\'],method=method,seed=seed),\'断点来源错配\')\n                e.check(m[\'checkpoint_sha256\']==e.sha(folder/\'best.pt\') and m[\'history_sha256\']==e.sha(folder/\'history.json\'),\'模型/历史哈希错误\')\n                row[\'status\']=\'trained_pending_prediction\'\n                if (folder/\'result.json\').exists():\n                    rr.append(export_prediction(ctx,m)); row[\'status\']=\'complete\'\n            inventory.append(row)\n        complete=len(rr)==3\n        records.extend(dict(r,eligible_for_comparison=complete) for r in rr)\n        if complete: families[method]=rr\n    return inventory,records,families\n\ndef train(ctx,max_new_groups=2):\n    e.check(isinstance(max_new_groups,int) and 0<=max_new_groups<=2,\'配置配额应为0、1或2\')\n    e.check(e.environment(ctx[\'device\'])==ctx[\'protocol\'][\'environment\'],\'本轮环境变化，不能混用断点\')\n    started=0\n    with e.exclusive(ctx[\'work\']):\n        for method in METHODS:\n            if method in collect(ctx)[2]: continue\n            if started>=max_new_groups: break\n            started+=1\n            try:\n                for seed in SEEDS: export_prediction(ctx,e.train_one(ctx,method,seed))\n            except (KeyboardInterrupt,RuntimeError) as ex:\n                e.save(ctx[\'work\']/f\'interruption_{uuid.uuid4().hex[:8]}.json\',dict(type=type(ex).__name__,message=str(ex)))\n                if isinstance(ex,KeyboardInterrupt) or \'out of memory\' in str(ex).lower(): return\n                raise\n        if len(collect(ctx)[2])==2:\n            e.freeze(ctx[\'work\']/\'selection.json\',dict(protocol_sha256=ctx[\'protocol_sha256\'],models=[e.read(ctx[\'work\']/f\'runs/{m}_{s}/done.json\') for m in METHODS for s in SEEDS]))\n\ndef report(ctx):\n    inv,records,families=collect(ctx); out=ctx[\'work\']/\'report\'; out.mkdir(exist_ok=True)\n    for method in [\'MM_CE\',\'MM_KD_raw\']: families[method]=[r for r in ctx[\'baseline\'] if r[\'method\']==method]\n    allrows=[dict(r,eligible_for_comparison=True) for r in ctx[\'baseline\']]+records\n    summary=[]\n    for method,rr in families.items():\n        e.check(len(rr)==3 and {r[\'seed\'] for r in rr}==set(SEEDS),\'仅完整三种子配置进入比较\')\n        summary.append(dict(method=method,n_seeds=3,**{k+\'_mean\':float(np.mean([r[k] for r in rr])) for k in METRICS},**{k+\'_seed_sd\':float(np.std([r[k] for r in rr],ddof=1)) for k in METRICS}))\n    deltas=[]; ds=[]\n    for a,b in [(\'Text_KD_raw\',\'Text_CE\'),(\'MM_CE\',\'Text_CE\'),(\'MM_KD_raw\',\'Text_KD_raw\')]:\n        if a not in families or b not in families: continue\n        aa={r[\'seed\']:r for r in families[a]}; bb={r[\'seed\']:r for r in families[b]}\n        rr=[dict(comparison=a+\' minus \'+b,seed=s,**{k:aa[s][k]-bb[s][k] for k in METRICS}) for s in SEEDS]\n        deltas+=rr\n        ds.append(dict(comparison=a+\' minus \'+b,n_seeds=3,**{k+\'_mean\':float(np.mean([r[k] for r in rr])) for k in METRICS},**{k+\'_seed_sd\':float(np.std([r[k] for r in rr],ddof=1)) for k in METRICS}))\n    table(out/\'all_6_conditions.csv\',inv); table(out/\'per_seed.csv\',allrows); table(out/\'summary.csv\',summary)\n    table(out/\'paired_deltas.csv\',deltas,[\'comparison\',\'seed\']+METRICS)\n    table(out/\'paired_delta_summary.csv\',ds,[\'comparison\',\'n_seeds\']+[k+s for s in [\'_mean\',\'_seed_sd\'] for k in METRICS])\n    missing=[r for r in inv if r[\'status\']!=\'complete\']\n    e.save(out/\'status.json\',dict(complete=not missing,completed=6-len(missing),expected=6,missing=missing,scope=ctx[\'protocol\'][\'scope\']))\n    e.atomic_bytes(out/\'README.md\',(\'文本单模态诊断；Teacher仍看图文，Student只看t。仅使用val_select，已用于选模，不是独立测试。\\n\'\n        \'差值方向见comparison；F1/Accuracy/ECE为0–1原值，乘100为百分数或百分点；NLL/ECE/Brier越低越好。SD为种子间样本标准差，不是配对置信区间。\\n\'\n        \'头参数并非完全相等；具体差异见protocol.json。结果不能单独证明跨模态推理能力。\\n\').encode(\'utf-8\'))\n    return out\n\ndef package(ctx):\n    report(ctx); work=ctx[\'work\']; dest=work/(\'Stage2_Hazel_text_only_results_\'+uuid.uuid4().hex[:8]+\'.zip\')\n    files=[work/\'protocol.json\']\n    if (work/\'selection.json\').exists(): files.append(work/\'selection.json\')\n    for folder in [\'code\',\'runs\',\'report\',\'baselines\',\'Stage2_reference\']:\n        files += [p for p in (work/folder).rglob(\'*\') if p.is_file() and p.suffix in [\'.json\',\'.csv\',\'.py\',\'.pt\',\'.md\'] and p.name!=\'state.pt\']\n    files+=list(work.glob(\'interruption_*.json\'))\n    with zipfile.ZipFile(dest,\'x\',compression=zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(set(files)): z.write(p,p.relative_to(work).as_posix())\n        z.writestr(\'PACKAGE_SHA256.json\',json.dumps({p.relative_to(work).as_posix():e.sha(p) for p in files},indent=2))\n    return dest\n'}
HASHES={'cache_audit.py': 'fc464669c08fd73dfd0449a92ba3b37e28595debc64aa32f0b11eac559cebd65', 'experiment.py': '1fd3c5ae620bcd091c75ca1b12854dd41f4324a63ba45249c3be53e1e66b6110', 'text_only.py': '79fbde4869ecdab89753faf7a49e8a9a0f5b65849f69083280808c8b5bff4731'}
CODE=WORK/'code'; CODE.mkdir(exist_ok=True)
for name,source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest()==HASHES[name]
    p=CODE/name
    if p.exists(): assert p.read_text(encoding='utf-8')==source, '代码改变，请保留旧WORK并新建输出目录'
    else: p.write_text(source,encoding='utf-8')
sys.path.insert(0,str(CODE))
for name in ['cache_audit','experiment','text_only']:
    spec=importlib.util.spec_from_file_location(name,CODE/(name+'.py'))
    module=importlib.util.module_from_spec(spec); sys.modules[name]=module;spec.loader.exec_module(module)
import text_only,experiment
print('文本实验代码已就绪')


## 4．核查输入、既有图文基线与分类头

核查训练6000条与val_select400条的特征/样本ID/标签/来源，以及划分间图像隔离。Teacher原始logits按sample_id对齐，类别顺序E/N/C。读取原始logits后只除tau一次，不读取已校准目标。

缓存结构必须是 `[v,t,abs(v−t),v*t]`，每段512维。文本输入严格切片 **x[:,512:1024]**，生成连续的float32数组；图像与两种交互项不进入Student。

固定文本头512→2048→3、图文头2048→512→3，ReLU/dropout=.1。参数量分别1,056,771与1,050,627；文本多6,144个（约0.585%），属于既定近似容量匹配，不宣称参数完全相等。两个文本设置使用相同头，不根据结果调整。

对既有图文CE(.0003)、原始KD(.0003,alpha=.75)三种子核查checkpoint和历史哈希，从逐条原始logits重算指标，并确认val_select样本/标签一致。新校准KD不进入这次三个主比较。图文模型、原始协议及预测快照保存在新WORK，原包保持不变。


In [ ]:
ctx=text_only.prepare(SOURCE,STAGE2_RESULTS,WORK)
print('输入：',ctx['protocol']['student_input'])
print('文本训练特征：',ctx['data']['train']['x'].shape)
print('分类头：',ctx['protocol']['heads'])
print('环境比对：',ctx['protocol']['environment_comparison'])
display(pd.DataFrame(ctx['baseline']))
print('本轮协议已锁定：',WORK/'protocol.json')


## 5．训练6条件

Text-only CE：CE(student_logits,gold)。

Text-only KD_raw：q=softmax(原始Jev logits/2)；L=.25×CE+.75×4×KL(q || softmax(student_logits/2))，KL使用batchmean。没有额外标签平滑。

30轮固定预算，按val_select原始学生Macro-F1最大、NLL最小、较早epoch选择checkpoint。评价时用原始学生logits，不除训练tau。每轮初始化随机流和批次顺序规则与Stage2一致；不同输入维度的头不能理解成权重逐个相同。

按完整三种子配置推进。断线后在同一环境与WORK从头重跑；完成模型自动跳过，未完成模型从最近epoch恢复。软中断或OOM会停止启动新配置并保留缺项；突然断连后可设MAX_NEW_GROUPS=0先整理结果。单种子不完整配置不进入均值和配对结论。

同一WORK只允许一个会话写入；只有确认旧会话停止后才能清理遗留ACTIVE.lock。


In [ ]:
try:
    text_only.train(ctx,max_new_groups=MAX_NEW_GROUPS)
finally:
    REPORT=text_only.report(ctx)
display(pd.read_csv(REPORT/'all_6_conditions.csv'))
print((REPORT/'status.json').read_text(encoding='utf-8'))


## 6．三个指定比较与完整结果表

1. Text_KD_raw − Text_CE：仅文本Student的蒸馏收益。
2. MM_CE − Text_CE：不蒸馏时图文输入相对文本输入的差异。
3. MM_KD_raw − Text_KD_raw：蒸馏后图文输入相对文本输入的差异。

全部按相同种子配对。两组都凑齐三个种子才输出该比较；全部完成时有9行逐种子差值与3行差值汇总。

输出Macro-F1、Accuracy、E/N/C各类F1、NLL、ECE10，以及沿用的Brier。ECE为10个等宽置信度分箱。所有指标为原值；F1/Accuracy/ECE乘100为百分数，差值乘100为百分点。NLL/ECE/Brier越低越好。三种子SD为样本标准差，不是配对置信区间。

Teacher仍看图片；文本KD收益不能解释为Teacher未使用视觉信息。图文−文本的差异也包含固定头结构的差异，不能单独证明跨模态推理能力。val_select已用于选模，这不是独立测试。无论结果高低全部保留，不挑最好种子，不扩大实验。


In [ ]:
REPORT=text_only.report(ctx)
display(pd.read_csv(REPORT/'per_seed.csv'))
display(pd.read_csv(REPORT/'summary.csv'))
display(pd.read_csv(REPORT/'paired_deltas.csv'))
display(pd.read_csv(REPORT/'paired_delta_summary.csv'))


## 7．打包结果

包含六个最佳文本checkpoint（完成多少保存多少）、每轮历史、逐条sample_id/gold/logits/概率/预测、配置、输入切片、头参数量、环境、原始教师软目标哈希、图文基线快照、全部结果表及PACKAGE_SHA256.json。逐条文本预测在runs/方法_种子/predictions.csv。

结果包不含原图、重复特征和优化器state.pt；断点保留WORK，恢复未完成训练时使用原WORK。report/status.json的complete=true、completed=6才表示6次训练完成。

本notebook与代码可由你上传Stage2_Hazel分支；训练结果另存新目录。芙芙的鱼负责的原始/校准KD学生后温度校准不在本notebook中重复执行。


In [ ]:
ZIP=text_only.package(ctx)
print('本阶段结果包：',ZIP)
print((WORK/'report/status.json').read_text(encoding='utf-8'))
